## Embeddings

For our previous example, we bin dey work wit high-dimensional bag-of-words vectors wey length na `vocab_size`, and we bin dey explicitly convert from low-dimensional positional representation vectors go sparse one-hot representation. Dis one-hot representation no memory-efficient, plus, each word dey treated independently from each oda, dat mean say one-hot encoded vectors no fit show any semantic similarity between words.

For dis unit, we go continue to explore **News AG** dataset. To begin, mek we load di data and grab some definitions from di previous notebook.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Wetin be embedding?

Di idea behind **embedding** na to represent words wit small-dimension dense vectors wey somehow dey show di meaning of di word. We go yan later how to build better word embeddings, but for now, mek we just think of embeddings as one way to reduce di dimension of word vector. 

So, embedding layer go take one word as input, den e go give output vector wey you specify `embedding_size`. For one kain level, e dey very similar to `Linear` layer, but instead make e take one-hot encoded vector, e go fit take word number as input.

If we use embedding layer as di first layer for our network, we fit change from bag-of-words to **embedding bag** model, where we go first convert every word inside our text to correct embedding, den we go compute some aggregation function on top of all those embeddings, like `sum`, `average` or `max`.  

![Image showing an embedding classifier for five sequence words.](../../../../../translated_images/pcm/embedding-classifier-example.b77f021a7ee67eee.webp)

Our classifier neural network go start wit embedding layer, den aggregation layer, den linear classifier for top am:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### How to handle variable sequence size

Because of dis architecture, minibatches wey go enter our network must be arranged in one kain way. For di last unit, wen we dey use bag-of-words, all BoW tensors for one minibatch get equal size `vocab_size`, no matter how long our text sequence really be. But wen we start to use word embeddings, we fit get different number of words for each text sample, and wen we join all those samples together for minibatches, we go need to add some padding.

You fit do dis by using di same method wey dey supply `collate_fn` function to di datasource:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Train embedding classifier

Now we don define correct dataloader, we fit train di model wit di training function we don define for di previous unit:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Note**: We just dey train for 25k records here (less than one full epoch) for time sake, but you fit continue to train, write function to train for plenty epochs, and try different learning rate parameters to get beta accuracy. You go fit reach accuracy of about 90%.


### EmbeddingBag Layer and Variable-Length Sequence Representation

For di previous architecture, we gats pad all di sequences make dem get di same length so dat we fit put dem for one minibatch. Dis no be di best way to represent variable length sequences - one oda way na to use **offset** vector, wey go hold all di offsets of di sequences wey dem pack for one big big vector.

![Image showing an offset sequence representation](../../../../../translated_images/pcm/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Note**: For di picture wey dey above, we show sequence of characters, but for our example we dey work with sequences of words. But di general principle of representing sequences with offset vector still dey di same.

To work with offset representation, we dey use [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) layer. E dey similar to `Embedding`, but e go take content vector and offset vector as input, and e still get averaging layer, wey fit be `mean`, `sum` or `max`.

Dis na di modified network wey dey use `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

To prepare di dataset for training, we need to provide one conversion function wey go prepare di offset vector:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Note, say unlike for all earlier examples, our network now dey accept two parameters: data vector and offset vector, wey be of different sizes. Similarly, our data loader still dey provide us with 3 values instead of 2: both text and offset vectors dem dey give as features. So, we need to little adjust our training function to handle that:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantic Embeddings: Word2Vec

For we previous example, di model embedding layer learn to map worl dem to vector representation, but dis representation no get plenty semantic meaning. E go beta if we fit learn dis kain vector representation, wey similar worl or synonyms go mean vectors wey dey close to each oda for some kind vector distance (ex. euclidian distance).

To do dat, we need pre-train our embedding model on big plenty text wen dem arrange am one kind way. One of di first ways to train semantic embeddings na wetin dem call [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). E get two main architectures wey dem dey use produce distributed representation of words:

 - **Continuous bag-of-words** (CBoW) — for dis architecture, we dey train di model to predict one word from di context wey surround am. Given di ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, di goal na to predict $W_0$ from $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** na opposite of CBoW. Di model dey use di surrounding context words for window predict di current word.

CBoW fast, but skip-gram slow pass, but e do better job representing words wey no too common.

![Image showing both CBoW and Skip-Gram algorithms to convert words to vectors.](../../../../../translated_images/pcm/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

To test word2vec embedding wey pre-train on Google News dataset, we fit use **gensim** library. Below we go find di words wey get close meaning to 'neural'

> **Note:** When you first create word vectors, downloading them fit take some time!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


We fit also calculate vector embeddings from di word, to take use am for training classification model (we only show di first 20 components of di vector to make am clear):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Beta tin wey dey semantical embeddings be say you fit manipulate vector encoding to change di semantics. For example, we fit ask make e find one word, wey im vector representation go near as e fit be to words *king* and *woman*, and far well-well from di word *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Both CBoW and Skip-Grams na “predictive” embeddings, because dem dey only use local contexts. Word2Vec no dey use global context at all. 

**FastText**, e build on top Word2Vec by dey learn vector representations for each word plus di character n-grams wey dey inside each word. Den na dem go average di values of di representations into one vector every training step. Even though dis one go make pre-training get more work, e still fit make word embeddings fit carry sub-word information. 

Another waya be **GloVe**, wey use di idea of co-occurrence matrix, e still dey use neural methods to break co-occurrence matrix into better and non linear word vectors.

You fit play wit di example by changing di embeddings to FastText and GloVe, because gensim fit support plenty different word embedding models.


## Usin Pre-Trained Embeddings for PyTorch

We fit change di example wey dey above to pre-populate di matrix for our embedding layer wit semantical embeddings, like Word2Vec. We gats consider say vocabularies of pre-trained embedding an our text corpus no go likely match, so we go initialize weights for di missing words wit random values:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Now mek we train our model. Make you sabi say the time wey e go take to train the model dea na plenti pass di previous example, because of di bigger embedding layer size, an so di number of parameters plenty well well. Plus, because of dis, we fit need train our model on more examples if we wan avoid overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

For our case, we no see big increase for accuracy, wey fit mean say the vocabularies different well well. 
To solve the palava of different vocabularies, we fit use one of this solution dem:
* Train word2vec model again with our vocabulary
* Load our dataset with the vocabulary from the pre-trained word2vec model. You fit specify the vocabulary wey you wan use when you dey load the dataset.

The second way e dey easier, specially becos PyTorch `torchtext` framework get support for embeddings for inside. For example, we fit make GloVe-based vocabulary like this:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary get the following basic operations:
* `vocab.stoi` dictionary dey allow us convert word go e dictionary index
* `vocab.itos` na de opposite - e dey convert number go word
* `vocab.vectors` na de array of embedding vectors, so to get de embedding of word `s` we need use `vocab.vectors[vocab.stoi[s]]`

Dis na example for how we fit manipulate embeddings to show equation **kind-man+woman = queen** (I get small adjust de coefficient to make am work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

To train di classifier wit dose embeddings, we fose need to encode our dataset wit GloVe vocabulary:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

As we don see for above, all vector embeddings dey store for `vocab.vectors` matrix. E make am super-easy to load those weights inside weights of embedding layer by just copying:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Now make we train our model and see if we go get beta results:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

One of di reasons we no dey see big increase for accuracy na because some words from our dataset no dey for di pre-trained GloVe vocabulary, so dem just dey basically ignored. To clear dis matter, we fit train our own embeddings for our dataset. 


## Contextual Embeddings

One key wahala wey traditional pretrained embedding dem get like Word2Vec na di problem of word sense disambiguation. Even if pretrained embeddings fit capture some meaning of words for context, every possible meaning of one word dey encoded inside di same embedding. Dis kin wahala fit spoil downstream models, because plenti words like di word 'play' get different meaning depending on di context wey dem dey use am.

For example, di word 'play' for these two different sentences mean different thing:
- I go see **play** for di theatre.
- John wan **play** wit him padi dem.

Di pretrained embeddings dem wey dey above dey represent both meaning of di word 'play' inside di same embedding. To solve dis kind limitation, we gots build embeddings based on **language model**, wey dem train with big big text corpus, and *know* how words fit join together for different contexts. To talk about contextual embeddings no dey inside di scope of dis tutorial, but we go turn back come dem when we dey yarn about language models for di next unit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
